# CONFIGURACIÓN INICIAL

## Librerías y API

In [60]:
import openrouteservice
import folium
from geopy.geocoders import Nominatim
import random
import numpy as np
from aux_functions import to_latlon, build_distance_matrix
from algoritmo_genetico import algoritmo_genetico
from algoritmo_aco import colonia_de_hormigas
from print_map import directions_geometry, indices_a_latlon, pintar, marcar_numeros
from algoritmo_hibrido import ga_aco_hibrido

# API Key de OpenRouteService
client = openrouteservice.Client(key='eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImFmYjllN2I5OWI2MzRmY2M4ZWMyYzU4YWE0NDAxOWM2IiwiaCI6Im11cm11cjY0In0=')

START_IDX = 0  # Índice del punto de inicio en la matriz de distancias
CERRAR_CICLO = False  # Indica si la ruta debe cerrar el ciclo volviendo al punto de inicio
SEED = 455 # Semilla para reproducibilidad

## Inicializar geolocalizador y obtener coordenadas

In [61]:
# Inicializar el geocodificador
geolocator = Nominatim(user_agent="route_optimizer")

# Lista para almacenar coordenadas
coordenadas = []

# Direcciones a geocodificar
direcciones = [
    "Puerta del Sol, Madrid",
    "Sagrada Familia, Barcelona",
    "Plaza del Pilar, Zaragoza",
    "Catedral de Sevilla, Sevilla",
    "Ciudad de las Artes y las Ciencias, Valencia",
    "La Alhambra, Granada, España",
    "Catedral de Santiago de Compostela, Santiago de Compostela",
    "Museo Guggenheim, Bilbao",
    "Mezquita-Catedral, Córdoba",
    "Plaza Mayor, Salamanca",
    "Casco Antiguo, Toledo",
    "Catedral de Burgos, Burgos",
    "Acueducto de Segovia, Segovia",
    "Parque Natural de las Bardenas Reales, Navarra",
    "Playa de la Concha, San Sebastián",
    "Puerto de Málaga, Málaga",
    "Cabo de Gata, Almería",
    "Ciudad Encantada, Cuenca",
    "Catedral de León, León",
    "Murallas de Ávila, Ávila"
]


# Geocodificar y almacenar coordenadas
for direccion in direcciones:
    loc = geolocator.geocode(direccion, timeout=20)
    if loc:
        print(f"{loc.address}")
        coordenadas.append((loc.longitude, loc.latitude))
    else:
        print(f"No se encontró: {direccion}")

print("\nCoordenadas:")
print(coordenadas)

COORDS_LATLON = to_latlon(coordenadas)

Puerta del Sol, Barrio de los Austrias, Sol, Centro, Madrid, Comunidad de Madrid, 28013, España
Basílica de la Sagrada Família, 401, Carrer de Mallorca, la Sagrada Família, Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, 08013, España
Plaza del Pilar, Olvés, Comunidad de Calatayud, Zaragoza, Aragón, España
Catedral de Sevilla, Calle Almirantazgo, El Arenal, Casco Antiguo, Sevilla, Andalucía, 41001, España
Ciutat de les Arts i les Ciències, 7, Avinguda del Professor López Piñero (Historiador de la Medicina), Ciutat de les Arts i de les Ciències, Quatre Carreres, València, Comarca de València, València / Valencia, Comunitat Valenciana, 46013, España
Museo de la Alhambra, Calle Real Alta, Realejo Alto, San Matías - Realejo, Centro, Granada, Comarca de la Vega de Granada, Granada, Andalucía, 18009, España
Catedral de Santiago de Compostela, Praza do Obradoiro, Cidade Vella, O Ensanche, Santiago de Compostela, Santiago, A Coruña, Galicia, 15705, España
Museo Guggenheim Bilbao, 2, Leh

## Construcción matriz de distancias D

In [62]:
D = build_distance_matrix(to_latlon(coordenadas))
D

array([[   0.     ,  620.02838,  244.72753,  534.01238,  364.71956,
         420.37363,  600.04925,  400.67353,  396.45506,  213.16728,
         128.71541,  247.56875,   91.3317 ,  359.38884,  453.38566,
         530.63519,  605.18844,  191.54019,  337.52231,  111.81093],
       [ 617.85806,    0.     ,  400.75428, 1006.53119,  362.16288,
         852.95638, 1083.79675,  606.15656,  873.86606,  831.08238,
         749.43819,  602.37863,  709.24681,  414.10778,  566.03388,
         976.68925,  792.25156,  576.55169,  774.86881,  729.72606],
       [ 243.46261,  400.73428,    0.     ,  780.33969,  275.41166,
         654.20113,  843.56888,  391.60313,  630.28256,  456.68697,
         375.04272,  251.16563,  334.85138,  153.88616,  303.70334,
         764.46269,  705.99206,  231.40872,  420.11359,  355.33059],
       [ 528.30831, 1009.29156,  769.87619,    0.     ,  666.59188,
         253.87347,  884.91806,  860.21938,  145.39144,  459.78138,
         407.33225,  707.31263,  602.33063,  

# ALGORITMO GENÉTICO

### Parámetros del algoritmo genético

- generaciones (int = 300)  
    - Número de iteraciones que ejecuta el algoritmo (ciclos de evolución).  
    - Cuanto más alto más oportunidades de mejorar la solución → mejor calidad final pero mayor tiempo de cómputo.  
    - Cuanto mas bajo menos tiempo pero riesgo de converger prematuramente a soluciones subóptimas.  

- pop_size (int = 100)  
    - Tamaño de la población (número de rutas/individuos por generación).  
    - Cuanto mas alto mayor diversidad, mejor exploración del espacio de soluciones → mayor probabilidad de mejores soluciones; aumenta coste computacional por generación.  
    - Cuanto mas bajo menos coste por generación pero menor diversidad y mayor riesgo de estancamiento.  

- pm (float = 0.2)  
    - Probabilidad de mutación por individuo o por gen (dependiendo de la implementación).  
    - Cuanto más alto mayor exploración aleatoria, ayuda a escapar de óptimos locales; valores muy altos rompen buenas soluciones (comportamiento casi aleatorio).  
    - Cuanto más bajo soluciones más estables y convergencia más rápida, pero riesgo de falta de exploración.  

- k (int = 5)  
    - Tamaño del torneo (número de individuos comparados para seleccionar un progenitor).  
    - Cuanto más alto aumenta presión selectiva (los mejores ganan con más probabilidad) → converge más rápido pero menor diversidad.  
    - Cuanto más bajo menor presión selectiva → más diversidad, búsqueda más exploratoria pero más lenta.  
    
- elitismo (int = 5)  
    - Número de mejores individuos que se copian directamente a la siguiente generación sin cambios.  
    - Cuanto más alto mejor preserva soluciones buenas garantizando no pérdida de lo mejor encontrado; puede reducir diversidad y fomentar estancamiento si es alto.  
    - Cuanto más bajo mayor renovación de la población, más exploración, pero riesgo de perder las mejores soluciones encontradas.  


In [63]:
# Algoritmo genético para optimizar la ruta
random.seed(SEED)
np.random.seed(SEED)
ruta_ga, coste_ga = algoritmo_genetico(D, generaciones=300, pop_size=100, pm=0.2, k=5, elitismo=5, start_idx=START_IDX, cerrar_ciclo=CERRAR_CICLO)

In [64]:
print("\nRuta optimizada (Algoritmo Genético):")
print(ruta_ga)
print(f"\nCoste total de la ruta: {coste_ga} km")


Ruta optimizada (Algoritmo Genético):
[0, 9, 6, 18, 11, 2, 13, 14, 7, 12, 19, 10, 8, 3, 15, 5, 16, 17, 4, 1]

Coste total de la ruta: 4504.57858 km


### Visualización del mapa

In [65]:
coords_ga  = indices_a_latlon(ruta_ga,  COORDS_LATLON)

geom_ga  = directions_geometry(coords_ga)
m = folium.Map(location=COORDS_LATLON[START_IDX], zoom_start=13)

pintar(m, geom_ga, "red")
marcar_numeros(m, coords_ga, "blue",
               margin_top=10, margin_left=-8,
               dx=0.00015, dy=-0.00010, alternate=True)

m.save('ruta_GA.html')

# ALGORITMO COLONIA DE HORMIGAS

### Parámetros del algoritmo ACO (Colonia de Hormigas)

- **n_hormigas (int = 50)**  
    - Número de hormigas que construyen soluciones en cada iteración.  
    - Más hormigas → mayor muestreo del espacio de soluciones y mejor probabilidad de encontrar buenas rutas; aumenta coste computacional linealmente por iteración.  
    - Menos hormigas → menos tiempo por iteración, pero menor diversidad y mayor riesgo de no explorar rutas prometedoras.

- **n_iteraciones (int = 300)**  
    - Número de ciclos de construcción + actualización de feromona.  
    - Más iteraciones permite que la información de feromona se acumule y estabilice en buenas rutas → calidad final probablemente mejor; tiempo total mayor.  
    - Menos iteraciones reduce tiempo pero puede dejar el algoritmo sin converger a soluciones buenas.

- **alpha (float = 1.0)**  
    - Peso de la feromona en la probabilidad de selección de la siguiente arista (importancia de la experiencia colectiva).  
    - Alpha alto → se favorecen fuertemente las aristas con mucha feromona (explotación). Si es demasiado alto hay riesgo de convergencia prematura a una solución subóptima.  
    - Alpha bajo → la feromona tiene menos influencia, se aumenta aleatoriedad/exploración.

- **beta (float = 5.0)**  
    - Peso de la heurística (normalmente 1/distancia) en la probabilidad de selección.  
    - Beta alto → se priorizan aristas más cortas (comportamiento más heurístico/greedy). Puede acelerar la obtención de buenas rutas pero reducir exploración.  
    - Beta bajo → la heurística influye poco, la elección es más guiada por feromona/azar.

- **rho (float = 0.1)**  
    - Tasa de evaporación de la feromona por iteración (fracción que se elimina).  
    - Rho alto (evaporación rápida) → la información antigua se pierde rápido, fomenta exploración y evita que soluciones tempranas dominen.  
    - Rho bajo (evaporación lenta) → las feromonas se acumulan y favorecen explotación de rutas encontradas, riesgo de estancamiento si hay mucho sesgo inicial.

- **Q (float = 100)**  
    - Constante usada para calcular la cantidad de feromona depositada (ej. depositar Q / coste_ruta).  
    - Q grande → refuerzos mayores por rutas buenas, acelerando la consolidación de caminos preferidos; si es demasiado grande puede conducir a dominancia rápida.  
    - Q pequeño → refuerzos débiles, aprendizaje más lento pero menos propenso a sobresaturación por soluciones iniciales.


In [66]:
# Algoritmo de colonia de hormigas para optimizar la ruta
random.seed(SEED)
np.random.seed(SEED)
ruta_aco, coste_aco = colonia_de_hormigas(D, n_hormigas=50, n_iteraciones=300, alpha=1.0, beta=5.0, rho=0.1, Q=100, start_idx=START_IDX, cerrar_ciclo=CERRAR_CICLO)

In [67]:
print("\nRuta optimizada (Colonia de Hormigas):")
print(ruta_aco)
print(f"\nCoste total de la ruta: {coste_aco} km")


Ruta optimizada (Colonia de Hormigas):
[0, 12, 19, 9, 6, 18, 11, 7, 14, 13, 2, 17, 10, 8, 3, 5, 15, 16, 4, 1]

Coste total de la ruta: 4195.18518 km


### Visualización del mapa

In [68]:
coords_aco  = indices_a_latlon(ruta_aco,  COORDS_LATLON)

geom_aco  = directions_geometry(coords_aco)
m = folium.Map(location=COORDS_LATLON[START_IDX], zoom_start=13)

pintar(m, geom_aco, "red")
marcar_numeros(m, coords_aco, "blue",
               margin_top=10, margin_left=-8,
               dx=0.00015, dy=-0.00010, alternate=True)

m.save('ruta_ACO.html')

## Mapa comparativo GA vs ACO

In [69]:
m = folium.Map(location=COORDS_LATLON[START_IDX], zoom_start=13)

pintar(m, geom_aco, "green")
marcar_numeros(m, coords_aco, "black",
               margin_top=-20, margin_left=8,
               dx=-0.00015, dy=0.00010, alternate=True)

pintar(m, geom_ga, "red")
marcar_numeros(m, coords_ga, "blue",
               margin_top=10, margin_left=-8,
               dx=0.00015, dy=-0.00010, alternate=True)

m.save('ruta_comparacion.html')

# ALGORITMO HÍBRIDO

In [70]:
random.seed(SEED)
np.random.seed(SEED)
ruta_h, coste_h, _ = ga_aco_hibrido(
    D,
    # parámetros GA
    n_runs_ga=10, generaciones_ga=100, pop_size=60, pm=0.2, k=3, elitismo=2,
    # parámetros ACO
    alpha=1.0, beta=5.0, rho=0.5, Q=100, n_hormigas=50, n_iteraciones=200,
    start_idx=START_IDX, cerrar_ciclo=CERRAR_CICLO
)

In [71]:
print("\nRuta optimizada (HIBRIDO):")
print(ruta_h)
print(f"\nCoste total de la ruta: {coste_h} km")


Ruta optimizada (HIBRIDO):
[0, 12, 19, 9, 6, 18, 11, 7, 14, 13, 2, 17, 10, 8, 3, 15, 5, 16, 4, 1]

Coste total de la ruta: 4095.6340600000003 km


### Visualización del mapa

In [72]:
coords_h  = indices_a_latlon(ruta_h,  COORDS_LATLON)

geom_h  = directions_geometry(coords_h)
m = folium.Map(location=COORDS_LATLON[START_IDX], zoom_start=13)

pintar(m, geom_h, "red")
marcar_numeros(m, coords_h, "blue",
               margin_top=10, margin_left=-8,
               dx=0.00015, dy=-0.00010, alternate=True)

m.save('ruta_HIBRIDO.html')

# MAPA FINAL 3 ALGORITMOS

In [73]:
m = folium.Map(location=COORDS_LATLON[START_IDX], zoom_start=13)

pintar(m, geom_aco, "green")
marcar_numeros(m, coords_aco, "green",
               margin_top=-20, margin_left=8,
               dx=-0.00015, dy=0.00010, alternate=True)

pintar(m, geom_ga, "red")
marcar_numeros(m, coords_ga, "red",
               margin_top=10, margin_left=-8,
               dx=0.00015, dy=-0.00010, alternate=True)

pintar(m, geom_h, "blue")
marcar_numeros(m, coords_h, "blue",
               margin_top=-10, margin_left=+8,
               dx=-0.00015, dy=-0.00010, alternate=True)

m.save('ruta_comparacion_3.html')